In [1]:
# %% 셀 1: 데이터 로드 + 임베딩 로드 + train/eval 분리
import os, json, random
import numpy as np
import torch
from tqdm import tqdm

POS_DIR = "./data/8_telop_position"
EMB_PATH = "./data/8_text_embeddings.pt"
GRID_W = 72
GRID_H = 128
EVAL_PER_CHANNEL = 5
SEED = 42

# ── 임베딩 로드 ──
text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")

# ── 데이터 로드 ──
json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

print(f"📁 파일 수: {len(json_paths):,}개")

channel_set = set()
samples = []

for channel, path in tqdm(json_paths, desc="로드"):
    with open(path, "r") as f:
        data = json.load(f)

    instances = data.get("instances", [])
    if not instances:
        continue

    channel_set.add(channel)
    duration = max(inst["end_sec"] for inst in instances)

    inst_list = []
    for inst in instances:
        # 좌표 이산화 (정수로)
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))

        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx,
            "y": gy,
            "w": gw,
            "h": gh,
        })

    samples.append({
        "channel": channel,
        "instances": inst_list,
        "duration": duration,
    })

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

# ── train/eval 분리 ──
rng = random.Random(SEED)
by_channel = {}
for s in samples:
    if s["channel"] not in by_channel:
        by_channel[s["channel"]] = []
    by_channel[s["channel"]].append(s)

train_samples, eval_samples = [], []
for ch, ch_samples in by_channel.items():
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

# ── 인스턴스 수 분포 ──
inst_counts = np.array([len(s["instances"]) for s in samples])
print(f"\n✅ 영상: {len(samples):,}개  |  채널: {len(channels)}개")
print(f"✅ train: {len(train_samples):,}  |  eval: {len(eval_samples):,}")
print(f"📊 인스턴스 수 — mean: {inst_counts.mean():.0f}  p99: {np.percentile(inst_counts, 99):.0f}  max: {inst_counts.max()}")

✅ 임베딩 로드: 1,448,729개  |  dim=1024
📁 파일 수: 66,400개


로드: 100%|██████████| 66400/66400 [01:00<00:00, 1104.72it/s]


✅ 영상: 59,876개  |  채널: 664개
✅ train: 56,556  |  eval: 3,320
📊 인스턴스 수 — mean: 62  p99: 419  max: 4251


In [2]:
# %% 셀 2: Dataset + DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 64
NUM_WORKERS = 32
MASK_RATIO = 1.0  # 학습 시 좌표 마스킹 비율 (1.0 = 전부 마스킹)


class BLTDataset(Dataset):
    def __init__(self, samples, channel2id, text2emb, mask_ratio=MASK_RATIO):
        self.samples = samples
        self.channel2id = channel2id
        self.text2emb = text2emb
        self.mask_ratio = mask_ratio

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        n = len(s["instances"])
        duration = max(s["duration"], 0.1)

        channel_id = self.channel2id.get(s["channel"], 0)
        channel_emb = self.text2emb.get(s["channel"], ZERO_EMB)

        text_embs = []
        cond_feats = []  # text_len, start_norm, end_norm, dur_norm
        gt_x, gt_y, gt_w, gt_h = [], [], [], []

        for inst in s["instances"]:
            text_embs.append(self.text2emb.get(inst["text"], ZERO_EMB))
            cond_feats.append([
                inst["text_len"] / 200.0,
                inst["start"] / duration,
                inst["end"] / duration,
                (inst["end"] - inst["start"]) / duration,
            ])
            gt_x.append(inst["x"])
            gt_y.append(inst["y"])
            gt_w.append(inst["w"] - 1)  # 1~72 → 0~71 (class index)
            gt_h.append(inst["h"] - 1)  # 1~128 → 0~127 (class index)

        # 마스킹: 학습 시 mask_ratio 비율로 좌표를 마스킹
        mask = torch.zeros(n, dtype=torch.bool)
        n_mask = max(1, int(n * self.mask_ratio))
        mask_idx = torch.randperm(n)[:n_mask]
        mask[mask_idx] = True

        return {
            "channel_id": channel_id,
            "channel_emb": channel_emb,                                    # (EMB_DIM,)
            "text_embs": torch.stack(text_embs),                           # (N, EMB_DIM)
            "cond_feats": torch.tensor(cond_feats, dtype=torch.float32),   # (N, 4)
            "gt_x": torch.tensor(gt_x, dtype=torch.long),                 # (N,)
            "gt_y": torch.tensor(gt_y, dtype=torch.long),                 # (N,)
            "gt_w": torch.tensor(gt_w, dtype=torch.long),                 # (N,)
            "gt_h": torch.tensor(gt_h, dtype=torch.long),                 # (N,)
            "coord_mask": mask,                                             # (N,) True=마스킹됨
            "n": n,
        }


def collate_fn(batch):
    max_n = max(b["n"] for b in batch)
    B = len(batch)

    channel_ids = torch.zeros(B, dtype=torch.long)
    channel_embs = torch.zeros(B, EMB_DIM)
    text_embs = torch.zeros(B, max_n, EMB_DIM)
    cond_feats = torch.zeros(B, max_n, 4)
    gt_x = torch.zeros(B, max_n, dtype=torch.long)
    gt_y = torch.zeros(B, max_n, dtype=torch.long)
    gt_w = torch.zeros(B, max_n, dtype=torch.long)
    gt_h = torch.zeros(B, max_n, dtype=torch.long)
    coord_mask = torch.zeros(B, max_n, dtype=torch.bool)
    seq_mask = torch.zeros(B, max_n, dtype=torch.bool)

    for i, b in enumerate(batch):
        n = b["n"]
        channel_ids[i] = b["channel_id"]
        channel_embs[i] = b["channel_emb"]
        text_embs[i, :n] = b["text_embs"]
        cond_feats[i, :n] = b["cond_feats"]
        gt_x[i, :n] = b["gt_x"]
        gt_y[i, :n] = b["gt_y"]
        gt_w[i, :n] = b["gt_w"]
        gt_h[i, :n] = b["gt_h"]
        coord_mask[i, :n] = b["coord_mask"]
        seq_mask[i, :n] = True

    return {
        "channel_id": channel_ids,
        "channel_emb": channel_embs,
        "text_embs": text_embs,
        "cond_feats": cond_feats,
        "gt_x": gt_x, "gt_y": gt_y, "gt_w": gt_w, "gt_h": gt_h,
        "coord_mask": coord_mask,   # True=이 인스턴스의 좌표가 마스킹됨
        "seq_mask": seq_mask,       # True=유효 인스턴스 (패딩 아님)
    }


train_ds = BLTDataset(train_samples, channel2id, text2emb, mask_ratio=1.0)
eval_ds = BLTDataset(eval_samples, channel2id, text2emb, mask_ratio=1.0)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)

batch = next(iter(train_loader))
print(f"✅ 배치 확인")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

✅ 배치 확인
  channel_id: torch.Size([64])
  channel_emb: torch.Size([64, 1024])
  text_embs: torch.Size([64, 382, 1024])
  cond_feats: torch.Size([64, 382, 4])
  gt_x: torch.Size([64, 382])
  gt_y: torch.Size([64, 382])
  gt_w: torch.Size([64, 382])
  gt_h: torch.Size([64, 382])
  coord_mask: torch.Size([64, 382])
  seq_mask: torch.Size([64, 382])


In [3]:
# %% 셀 3: BLT 모델 정의
D_MODEL = 256
N_HEADS = 8
N_LAYERS = 6
D_FF = 512
DROPOUT = 0.1

# 좌표 vocab 크기
N_X = GRID_W      # 72
N_Y = GRID_H      # 128
N_W = GRID_W      # 72 (1~72 → 0~71)
N_H = GRID_H      # 128 (1~128 → 0~127)


class BLTLayoutModel(nn.Module):
    def __init__(self, n_channels, emb_dim=EMB_DIM, d_model=D_MODEL,
                 n_heads=N_HEADS, n_layers=N_LAYERS, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()

        # ── 조건 인코딩 ──
        self.channel_id_emb = nn.Embedding(n_channels, d_model)
        self.channel_text_proj = nn.Linear(emb_dim, d_model)
        self.text_proj = nn.Linear(emb_dim, d_model)
        self.cond_proj = nn.Linear(4, d_model)

        # ── 좌표 임베딩 (마스킹 안 된 인스턴스용) ──
        self.x_emb = nn.Embedding(N_X + 1, d_model // 4)  # +1 for MASK token
        self.y_emb = nn.Embedding(N_Y + 1, d_model // 4)
        self.w_emb = nn.Embedding(N_W + 1, d_model // 4)
        self.h_emb = nn.Embedding(N_H + 1, d_model // 4)

        self.MASK_X = N_X  # mask token index
        self.MASK_Y = N_Y
        self.MASK_W = N_W
        self.MASK_H = N_H

        self.coord_proj = nn.Linear(d_model, d_model)

        # ── 토큰 결합 ──
        self.token_norm = nn.LayerNorm(d_model)

        # ── Bidirectional Transformer ──
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # ── 좌표 예측 heads ──
        self.head_x = nn.Linear(d_model, N_X)
        self.head_y = nn.Linear(d_model, N_Y)
        self.head_w = nn.Linear(d_model, N_W)
        self.head_h = nn.Linear(d_model, N_H)

    def encode_coords(self, gt_x, gt_y, gt_w, gt_h, coord_mask):
        """좌표를 임베딩. 마스킹된 위치는 MASK 토큰 사용."""
        B, N = gt_x.shape

        x_ids = gt_x.clone()
        y_ids = gt_y.clone()
        w_ids = gt_w.clone()
        h_ids = gt_h.clone()

        # 마스킹된 위치에 MASK 토큰 삽입
        x_ids[coord_mask] = self.MASK_X
        y_ids[coord_mask] = self.MASK_Y
        w_ids[coord_mask] = self.MASK_W
        h_ids[coord_mask] = self.MASK_H

        # 각 좌표 임베딩 후 concat
        xe = self.x_emb(x_ids)  # (B, N, d//4)
        ye = self.y_emb(y_ids)
        we = self.w_emb(w_ids)
        he = self.h_emb(h_ids)

        coord = torch.cat([xe, ye, we, he], dim=-1)  # (B, N, d)
        return self.coord_proj(coord)

    def forward(self, channel_id, channel_emb, text_embs, cond_feats,
                gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask):
        """
        Returns: logits_x (B,N,72), logits_y (B,N,128), logits_w (B,N,72), logits_h (B,N,128)
        """
        B, N, _ = text_embs.shape

        # ① 조건 인코딩
        ch_id = self.channel_id_emb(channel_id).unsqueeze(1).expand(-1, N, -1)
        ch_text = self.channel_text_proj(channel_emb).unsqueeze(1).expand(-1, N, -1)
        text = self.text_proj(text_embs)
        cond = self.cond_proj(cond_feats)

        # ② 좌표 임베딩 (마스킹 적용)
        coord = self.encode_coords(gt_x, gt_y, gt_w, gt_h, coord_mask)

        # ③ 토큰 합성
        tokens = ch_id + ch_text + text + cond + coord
        tokens = self.token_norm(tokens)

        # ④ Bidirectional Transformer
        padding_mask = ~seq_mask
        out = self.transformer(tokens, src_key_padding_mask=padding_mask)

        # ⑤ 좌표 예측
        logits_x = self.head_x(out)  # (B, N, 72)
        logits_y = self.head_y(out)  # (B, N, 128)
        logits_w = self.head_w(out)  # (B, N, 72)
        logits_h = self.head_h(out)  # (B, N, 128)

        return logits_x, logits_y, logits_w, logits_h


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BLTLayoutModel(n_channels=len(channels)).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🖥️  Device: {device}")
print(f"📐 파라미터: {n_params:,}")

🖥️  Device: cuda
📐 파라미터: 4,053,648


In [ ]:
# %% 셀 4: 학습
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 50
LR = 1e-4
SAVE_DIR = "./model/8_layout_blt_model"
os.makedirs(SAVE_DIR, exist_ok=True)


def gaussian_soft_label(gt, n_classes, sigma=2.0):
    """GT 주변 가우시안 soft label 생성"""
    device = gt.device
    arange = torch.arange(n_classes, device=device).float()  # (C,)
    gt_f = gt.float().unsqueeze(-1)  # (..., 1)
    label = torch.exp(-0.5 * ((arange - gt_f) / sigma) ** 2)  # (..., C)
    label = label / label.sum(dim=-1, keepdim=True)  # 정규화
    return label


def compute_loss(logits_x, logits_y, logits_w, logits_h,
                 gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask):
    valid = coord_mask & seq_mask

    if valid.sum() == 0:
        return torch.tensor(0.0, device=logits_x.device, requires_grad=True)

    # soft label 생성
    soft_x = gaussian_soft_label(gt_x[valid], N_X, sigma=2.0)
    soft_y = gaussian_soft_label(gt_y[valid], N_Y, sigma=3.0)
    soft_w = gaussian_soft_label(gt_w[valid], N_W, sigma=2.0)
    soft_h = gaussian_soft_label(gt_h[valid], N_H, sigma=1.0)

    # KL divergence (soft label과의 cross-entropy)
    loss_x = F.cross_entropy(logits_x[valid], soft_x)
    loss_y = F.cross_entropy(logits_y[valid], soft_y)
    loss_w = F.cross_entropy(logits_w[valid], soft_w)
    loss_h = F.cross_entropy(logits_h[valid], soft_h)

    return (loss_x + loss_y + loss_w + loss_h) / 4

@torch.no_grad()
def compute_metrics(logits_x, logits_y, logits_w, logits_h,
                    gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask):
    valid = coord_mask & seq_mask

    if valid.sum() == 0:
        return {"acc_x": 0, "acc_y": 0, "acc_w": 0, "acc_h": 0,
                "mae_x": 0, "mae_y": 0, "mae_w": 0, "mae_h": 0}

    pred_x = logits_x[valid].argmax(dim=-1)
    pred_y = logits_y[valid].argmax(dim=-1)
    pred_w = logits_w[valid].argmax(dim=-1)
    pred_h = logits_h[valid].argmax(dim=-1)

    return {
        "acc_x": (pred_x == gt_x[valid]).float().mean().item(),
        "acc_y": (pred_y == gt_y[valid]).float().mean().item(),
        "acc_w": (pred_w == gt_w[valid]).float().mean().item(),
        "acc_h": (pred_h == gt_h[valid]).float().mean().item(),
        "mae_x": (pred_x - gt_x[valid]).abs().float().mean().item(),
        "mae_y": (pred_y - gt_y[valid]).abs().float().mean().item(),
        "mae_w": (pred_w - gt_w[valid]).abs().float().mean().item(),
        "mae_h": (pred_h - gt_h[valid]).abs().float().mean().item(),
    }


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_eval_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    # ── train ──
    model.train()
    train_loss_sum, train_batches = 0, 0
    for batch in tqdm(train_loader, desc=f"[{epoch}/{EPOCHS}] train", leave=False):
        channel_id = batch["channel_id"].to(device)
        channel_emb = batch["channel_emb"].to(device)
        text_embs = batch["text_embs"].to(device)
        cond_feats = batch["cond_feats"].to(device)
        gt_x = batch["gt_x"].to(device)
        gt_y = batch["gt_y"].to(device)
        gt_w = batch["gt_w"].to(device)
        gt_h = batch["gt_h"].to(device)
        coord_mask = batch["coord_mask"].to(device)
        seq_mask = batch["seq_mask"].to(device)

        lx, ly, lw, lh = model(channel_id, channel_emb, text_embs, cond_feats,
                                gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask)
        loss = compute_loss(lx, ly, lw, lh, gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    # ── eval ──
    model.eval()
    eval_loss_sum, eval_batches = 0, 0
    all_metrics = {k: [] for k in ["acc_x", "acc_y", "acc_w", "acc_h",
                                    "mae_x", "mae_y", "mae_w", "mae_h"]}

    with torch.no_grad():
        for batch in eval_loader:
            channel_id = batch["channel_id"].to(device)
            channel_emb = batch["channel_emb"].to(device)
            text_embs = batch["text_embs"].to(device)
            cond_feats = batch["cond_feats"].to(device)
            gt_x = batch["gt_x"].to(device)
            gt_y = batch["gt_y"].to(device)
            gt_w = batch["gt_w"].to(device)
            gt_h = batch["gt_h"].to(device)
            coord_mask = batch["coord_mask"].to(device)
            seq_mask = batch["seq_mask"].to(device)

            lx, ly, lw, lh = model(channel_id, channel_emb, text_embs, cond_feats,
                                    gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask)
            loss = compute_loss(lx, ly, lw, lh, gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask)
            metrics = compute_metrics(lx, ly, lw, lh, gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask)

            eval_loss_sum += loss.item()
            eval_batches += 1
            for k, v in metrics.items():
                all_metrics[k].append(v)

    scheduler.step()

    train_loss = train_loss_sum / max(train_batches, 1)
    eval_loss = eval_loss_sum / max(eval_batches, 1)
    avg_m = {k: np.mean(v) for k, v in all_metrics.items()}
    lr_now = optimizer.param_groups[0]["lr"]

    print(
        f"[{epoch:>3}/{EPOCHS}]  "
        f"train={train_loss:.4f}  eval={eval_loss:.4f}  "
        f"mae: x={avg_m['mae_x']:.2f} y={avg_m['mae_y']:.2f} "
        f"w={avg_m['mae_w']:.2f} h={avg_m['mae_h']:.2f}  "
        f"acc: x={avg_m['acc_x']:.3f} y={avg_m['acc_y']:.3f} "
        f"w={avg_m['acc_w']:.3f} h={avg_m['acc_h']:.3f}  "
        f"lr={lr_now:.2e}"
    )

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "eval_loss": eval_loss,
        "eval_metrics": avg_m,
        "channel2id": channel2id,
    }, os.path.join(SAVE_DIR, f"epoch_{epoch:03d}.pt"))

    if eval_loss < best_eval_loss:
        best_eval_loss = eval_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "eval_loss": eval_loss,
            "eval_metrics": avg_m,
            "channel2id": channel2id,
        }, os.path.join(SAVE_DIR, "best.pt"))
        print(f"   💾 best 갱신 (eval_loss={eval_loss:.4f})")

print(f"\n✅ 완료. Best eval_loss={best_eval_loss:.4f}")

/home/taeyoung/miniconda3/envs/chi/lib/python3.12/site-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


[  1/50]  train=3.7869  eval=3.7426  mae: x=17.30 y=31.45 w=12.38 h=2.31  acc: x=0.031 y=0.020 w=0.057 h=0.187  lr=9.99e-05
   💾 best 갱신 (eval_loss=3.7426)


[  2/50]  train=3.6947  eval=3.6812  mae: x=16.47 y=31.34 w=13.07 h=2.28  acc: x=0.049 y=0.029 w=0.062 h=0.222  lr=9.96e-05
   💾 best 갱신 (eval_loss=3.6812)


[  3/50]  train=3.6348  eval=3.6307  mae: x=15.64 y=27.98 w=12.79 h=2.22  acc: x=0.048 y=0.036 w=0.062 h=0.238  lr=9.91e-05
   💾 best 갱신 (eval_loss=3.6307)


[4/50] train:  91%|█████████ | 801/884 [01:08<00:05, 15.36it/s]